# 168. 模型合并：Task Arithmetic、TIES 与 DARE 怎样从零实现并验收？

> **面试问题：多个微调模型不再训练能否直接合并？参数兼容、符号冲突、trim、drop-rescale 和 scale 搜索怎样做？**

## 先给结论

模型合并是在同一 base 上组合 task vector，不是任意 checkpoint 求平均。Task Arithmetic 简单但会叠加冗余与符号冲突；TIES 先裁剪小增量、选举主符号再只合并同符号值；DARE 随机丢弃 delta 并按保留率重缩放。最终优劣只能由多任务与安全回归决定。

## 推荐的回答主线

1. 先验证架构、参数名/shape、tokenizer、base checkpoint 和参数语义完全兼容。
2. 把每个微调模型写成 `base + delta_i`，实现线性 task arithmetic 并量化冲突。
3. 实现 TIES 的 trim/elect/disjoint merge 与 DARE 的 drop-rescale，说明超参数含义。
4. 在独立验证集搜索 scale/密度，检查单任务保持、组合收益、安全、漂移和可回滚制品。

## 本 Notebook 的实现边界

小向量只展示逐参数机制；真实 Transformer 还要处理 embedding/head tie、norm、buffer、MoE expert、量化权重与 tokenizer。参数空间指标不能替代下游生成评测。

## 一手资料

- [Task Arithmetic](https://arxiv.org/abs/2212.04089)
- [TIES-Merging](https://arxiv.org/abs/2306.01708)
- [DARE](https://arxiv.org/abs/2311.03099)


In [ ]:
import hashlib
import itertools
import json
import math
from dataclasses import dataclass, asdict

import numpy as np

# 受控 delta 同时包含共识、异号冲突和近零冗余坐标。
rng = np.random.default_rng(168)
base = np.array([1.0, -0.5, 0.2, 0.0, 2.0, -1.0, 0.4, 0.8])
deltas = np.array([
    [0.8, 0.02, -0.7, 0.1, 0.0, 0.5, -0.3, 0.04],
    [0.6, -0.01, 0.5, 0.0, 0.2, 0.4, 0.2, -0.03],
    [-0.4, 0.03, -0.6, 0.2, -0.1, 0.3, 0.1, 0.02],
])

assert deltas.ndim == 2
assert deltas.shape[1] == base.size
assert np.any((deltas > 0).any(0) & (deltas < 0).any(0))


## 1. 兼容性合同：同名、同 shape、同 base 只是最低要求

两个模型即使层数相同，若 tokenizer、词表顺序、RoPE、chat template 或 base 不同，逐元素相加也没有语义保证。先让 manifest 精确匹配，再读取参数；不应靠文件名猜 base。


In [ ]:
@dataclass(frozen=True)
class ModelManifest:
    architecture: str
    base_hash: str
    tokenizer_hash: str
    parameter_shapes: tuple

def compatible(manifests):
    return len(set(manifests)) == 1

# 完全相同才兼容；tokenizer 或 shape 改变必须拒绝。
manifest = ModelManifest("tiny-decoder-v1", "base-abc", "tok-xyz", ((8,),))
assert compatible([manifest, manifest])
assert not compatible([manifest, ModelManifest("tiny-decoder-v1", "base-abc", "tok-other", ((8,),))])
assert not compatible([manifest, ModelManifest("tiny-decoder-v1", "base-abc", "tok-xyz", ((9,),))])


## 2. Task Arithmetic：在 delta 空间组合，而不是平均绝对权重

定义 `delta_i = theta_i - theta_base`，合并为 `theta_base + lambda * sum_i delta_i`。如果参与模型权重不同，可加任务系数；lambda 必须在 holdout 搜索，任务数增加时不能机械保持 1。


In [ ]:
def task_arithmetic(base_vector, task_deltas, scale=1.0, task_weights=None):
    if task_weights is None:
        task_weights = np.ones(len(task_deltas))
    combined = np.sum(task_deltas * np.asarray(task_weights)[:, None], axis=0)
    return base_vector + scale * combined

# 单任务 scale=1 应复原对应微调模型；scale=0 应回到 base。
merged_linear = task_arithmetic(base, deltas, scale=0.5)
assert np.allclose(task_arithmetic(base, deltas[:1], 1.0), base + deltas[0])
assert np.allclose(task_arithmetic(base, deltas, 0.0), base)
assert merged_linear.shape == base.shape


## 3. 冲突诊断：异号比例和抵消量比合并后范数更有解释力

坐标上同时出现正负大增量时，直接求和会相互抵消；大量小增量则可能累积成噪声。可报告活跃任务数、符号熵、`sum(abs(delta))-abs(sum(delta))` 抵消量，并按层切片定位。


In [ ]:
def conflict_report(task_deltas, eps=1e-8):
    positive = (task_deltas > eps).any(axis=0)
    negative = (task_deltas < -eps).any(axis=0)
    conflict = positive & negative
    cancellation = np.abs(task_deltas).sum(axis=0) - np.abs(task_deltas.sum(axis=0))
    return conflict, cancellation

# 冲突坐标必有正的抵消量，无冲突同号坐标抵消量应近零。
conflict, cancellation = conflict_report(deltas)
assert conflict.any()
assert np.all(cancellation[conflict] > 0)
assert np.all(cancellation[~conflict] >= -1e-12)


## 4. TIES 第一步 Trim：按每个任务保留大幅 delta

Trim 不是按全局统一阈值，而可按每个 task 的绝对值分位或 top-k 保留；保留密度控制稀疏度。相等值需要稳定 tie-break，否则不同平台可能生成不同 mask。


In [ ]:
def trim_topk(task_deltas, density):
    kept = np.zeros_like(task_deltas)
    k = max(1, int(math.ceil(task_deltas.shape[1] * density)))
    for row, delta in enumerate(task_deltas):
        order = np.argsort(-np.abs(delta), kind="stable")[:k]
        kept[row, order] = delta[order]
    return kept

# 每个任务恰保留 ceil(density*d) 个坐标，值来自原 delta。
trimmed = trim_topk(deltas, density=0.5)
expected_k = math.ceil(base.size * 0.5)
assert np.all((trimmed != 0).sum(axis=1) == expected_k)
assert np.all((trimmed == 0) | (trimmed == deltas))
assert np.linalg.norm(trimmed) <= np.linalg.norm(deltas) + 1e-12


## 5. TIES Elect + Disjoint Merge：只聚合同主符号的活跃值

Elect 可按各任务 delta 之和的符号决定主方向；随后只选与主符号一致的非零值做均值。全零坐标必须返回 0，不能产生 NaN。论文实现细节可能有变体，面试时要把自己的规则说清。


In [ ]:
def ties_merge(task_deltas, density=0.5):
    trimmed = trim_topk(task_deltas, density)
    elected = np.sign(trimmed.sum(axis=0))
    merged = np.zeros(trimmed.shape[1])
    for j, sign in enumerate(elected):
        selected = trimmed[:, j][np.sign(trimmed[:, j]) == sign] if sign != 0 else np.array([])
        merged[j] = selected.mean() if selected.size else 0.0
    return merged, trimmed, elected

# 合并向量有限；非零坐标符号服从 elect；全零输入保持全零。
ties_delta, ties_trimmed, elected = ties_merge(deltas, density=0.5)
assert np.isfinite(ties_delta).all()
assert np.all(np.sign(ties_delta[ties_delta != 0]) == elected[ties_delta != 0])
assert np.allclose(ties_merge(np.zeros_like(deltas), 0.5)[0], 0.0)


## 6. DARE：随机 drop 后除以保留率，保持期望不变

以 drop rate p 丢弃 delta 坐标，保留项除以 `1-p`。单次 realization 会有方差，只在随机 mask 的期望上等于原 delta；p 太高或极端坐标都可能破坏模型。部署必须记录 seed/mask 或物化合并权重。


In [ ]:
def dare(delta, drop_rate, generator):
    if not 0 <= drop_rate < 1:
        raise ValueError("drop_rate 必须在 [0,1)")
    keep = generator.random(delta.shape) >= drop_rate
    return delta * keep / (1.0 - drop_rate), keep

# 多次采样均值应接近原 delta；每次未保留坐标严格为零。
draws = []
for seed in range(3000):
    sampled, keep = dare(deltas[0], 0.4, np.random.default_rng(seed))
    assert np.all(sampled[~keep] == 0)
    draws.append(sampled)
draw_mean = np.mean(draws, axis=0)
assert np.allclose(draw_mean, deltas[0], atol=0.04)
assert dare(deltas[0], 0.0, np.random.default_rng(1))[1].all()


## 7. DARE + TIES：先稀疏重缩放，再消解符号冲突

一种组合是对各 task delta 做 DARE，再执行 TIES。顺序、drop rate、density、任务权重和随机 seed 都是 recipe 的一部分；不要只发布一个模型文件而丢失生成过程。


In [ ]:
def dare_ties(task_deltas, drop_rate, density, seed):
    generator = np.random.default_rng(seed)
    sparse = np.stack([dare(delta, drop_rate, generator)[0] for delta in task_deltas])
    merged, _, _ = ties_merge(sparse, density)
    return merged, sparse

# 固定 seed 必须可复现，不同 seed 通常产生不同 realization，输出形状保持全局参数轴。
combo_delta, sparse = dare_ties(deltas, 0.25, 0.5, seed=42)
combo_again, _ = dare_ties(deltas, 0.25, 0.5, seed=42)
combo_other, _ = dare_ties(deltas, 0.25, 0.5, seed=43)
assert np.array_equal(combo_delta, combo_again)
assert not np.array_equal(sparse, dare_ties(deltas, 0.25, 0.5, seed=43)[1])
assert combo_delta.shape == base.shape


## 8. Scale 搜索与制品门禁：用多任务 holdout 决定，不用参数范数决定

下面用受控二次目标演示 Pareto 式选择：平均任务损失最小且每个任务不超过回归上限。真实发布还要加入安全、语言、长上下文、校准和成本切片，并与单任务模型及 base 成对比较。


In [ ]:
def task_losses(candidate, desired_vectors):
    return np.array([np.mean((candidate - desired) ** 2) for desired in desired_vectors])

scales = np.linspace(0.0, 1.2, 25)
desired = base[None, :] + deltas
candidates = [base + scale * ties_delta for scale in scales]
loss_table = np.stack([task_losses(candidate, desired) for candidate in candidates])
best_index = int(loss_table.mean(axis=1).argmin())
best_scale, best_model = scales[best_index], candidates[best_index]

recipe = {"method": "ties", "density": 0.5, "scale": float(best_scale), "base": "base-abc"}
digest = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode() + best_model.tobytes()).hexdigest()
# 搜索结果必须来自候选网格，摘要稳定，且所选平均 loss 不差于 base(scale=0)。
assert best_scale in scales
assert loss_table[best_index].mean() <= loss_table[0].mean()
assert len(digest) == 64


## 面试收束：怎样把实现讲成工程答案

建议按“目标与约束 → 数据/张量合同 → 核心公式 → 正确性 oracle → 性能与安全边界 → 发布门禁”作答。Notebook 里的小张量和受控状态机只证明机制成立，不等于真实集群吞吐、真实模型质量或生产安全性。上线前还要补齐目标硬件 profiling、故障注入、分布式一致性、真实数据切片、权限审计、版本化制品和回滚演练。

可继续追问：规模扩大后哪个状态最贵？哪条等价性可作为回归测试？输入或版本变化时怎样拒绝静默错误？指标改善是否只是成本、数据污染或评测器偏差造成的？
